# 02: Tiny GenImage inventory (Step 4)

Downloads [Tiny GenImage](https://huggingface.co/datasets/TheKernel01/Tiny-GenImage) to Colab's local disk and catalogues every image: split, label, generator, format, size. Then it checks for format and size biases at scale.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import io
import numpy as np
import pandas as pd
from PIL import Image

ROOT = "/content/drive/MyDrive/minor-project"
Path(ROOT, "results").mkdir(exist_ok=True)

Mounted at /content/drive


In [2]:
!pip install -q datasets
from datasets import load_dataset, Image as HFImage

ds = load_dataset("TheKernel01/Tiny-GenImage", cache_dir="/content/hf_cache")
print(ds)

README.md:   0%|          | 0.00/4.35k [00:00<?, ?B/s]

data/train-00000-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  477MB            

data/train-00000-of-00014.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00014.parquet: reconstructing file:   0%|          |  0.00B /  475MB            

data/train-00001-of-00014.parquet: downloading bytes:           |  0.00B            

...

data/validation-00003-of-00004.parquet: reconstructing file:   0%|          |  0.00B /  425MB            

data/validation-00003-of-00004.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/28000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'generator'],
        num_rows: 28000
    })
    validation: Dataset({
        features: ['image', 'label', 'generator'],
        num_rows: 7000
    })
})


In [3]:
GENERATORS = ds["train"].features["generator"].names
LABELS = ds["train"].features["label"].names

rows = []
for split in ["train", "validation"]:
    part = ds[split].cast_column("image", HFImage(decode=False))
    for i, example in enumerate(part):
        raw = example["image"]["bytes"]
        img = Image.open(io.BytesIO(raw))
        rows.append({
            "split": split,
            "index": i,
            "label": LABELS[example["label"]],
            "generator": GENERATORS[example["generator"]],
            "format": img.format,
            "mode": img.mode,
            "width": img.size[0],
            "height": img.size[1],
            "file_kb": round(len(raw) / 1024),
        })

inventory = pd.DataFrame(rows)
inventory.to_csv(Path(ROOT, "results", "03_tiny_genimage_inventory.csv"), index=False)
print(len(inventory), "images catalogued")

35000 images catalogued


In [4]:
print("Images per generator and split:")
display(pd.crosstab(inventory["generator"], inventory["split"]))

print("\nFile format per generator:")
display(pd.crosstab(inventory["generator"], inventory["format"]))

print("\nTypical (median) size per generator:")
display(inventory.groupby("generator")[["width", "height", "file_kb"]].median())

says_fake = inventory["format"] == "PNG"
accuracy = (np.where(says_fake, "fake", "real") == inventory["label"]).mean()
print(f"\nCheat rule 'PNG = fake' on the whole dataset: {accuracy:.1%}")

Images per generator and split:


split,train,validation
generator,,
ADM,2000,500
BigGAN,2000,500
GLIDE,2000,500
Midjourney,2000,500
Real,14000,3500
SD15,2000,500
VQDM,2000,500
Wukong,2000,500



File format per generator:


format,JPEG,PNG
generator,,
ADM,0,2500
BigGAN,0,2500
GLIDE,0,2500
Midjourney,0,2500
Real,17500,0
SD15,0,2500
VQDM,0,2500
Wukong,0,2500



Typical (median) size per generator:


,width,height,file_kb
generator,,,
ADM,256.0,256.0,117.0
BigGAN,128.0,128.0,29.0
GLIDE,256.0,256.0,75.0
Midjourney,1024.0,1024.0,1211.0
Real,500.0,375.0,109.0
SD15,512.0,512.0,448.0
VQDM,256.0,256.0,109.0
Wukong,512.0,512.0,454.0



Cheat rule 'PNG = fake' on the whole dataset: 100.0%
